# Ionospheric Scintillation Analyzer -- Usage Example

Full pipeline: load PM6 data, clean signal, bandpass filter, CWT spectrogram.

Uses the `core/` modules of the analyzer.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, "..")

from core.parsers import load_pm6_data, parse_regi_with_time, build_observation_sessions
from core.signal_processing import clean_and_smooth_signal, compute_cwt_spectrogram
from core.config import DEFAULT_WINDOW_SIZE, DEFAULT_N_SIGMAS, CWT_NV, PCHIP_FACTOR
from scipy.signal import butter, sosfiltfilt

plt.rcParams.update({"figure.dpi": 120})

## 1. Load PM6 Data
Set `PM6_FILE` to your `.pm6` file path.

In [ ]:
PM6_FILE = "../data/your_file.pm6"  # <-- change this
CHANNEL  = "P1_20A"                 # receiver channel

df, fs, start_dt = load_pm6_data(PM6_FILE)
n = len(df)
print(f"Loaded {n} samples  |  fs={fs} Hz  |  Start: {start_dt}")
df.head()

## 2. Parse Observation Sessions from REGI Log

In [ ]:
REGI_FILE = "../data/your_log.regi"  # <-- change this

events   = parse_regi_with_time(REGI_FILE, start_dt)
sessions = build_observation_sessions(events, df["Time_sec"].values)

for s in sessions:
    dur = (s["end"] - s["start"]) / 60
    tgt = s["target"]
    print(f"{tgt:20s}  {dur:.1f} min")

## 3. Clean Signal and Bandpass Filter
Hampel outlier removal + Savitzky-Golay smoothing, then a Butterworth bandpass.

In [ ]:
# Select one session (change index for a different source transit)
session = sessions[0]
s_idx = int(df["Time_sec"].searchsorted(session["start"]))
e_idx = int(df["Time_sec"].searchsorted(session["end"]))
time_s = df["Time_sec"].values[s_idx:e_idx]
signal = df[CHANNEL].values[s_idx:e_idx]

cleaned = clean_and_smooth_signal(signal, DEFAULT_WINDOW_SIZE, DEFAULT_N_SIGMAS)

LOWCUT, HIGHCUT = 0.1, 0.5  # Hz
sos = butter(4, [LOWCUT, HIGHCUT], btype="bandpass", fs=fs, output="sos")
scintillations = sosfiltfilt(sos, cleaned)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(time_s/3600, signal,         color="steelblue", lw=0.5)
axes[0].set(ylabel="Amplitude", title="Raw Signal"); axes[0].grid(alpha=0.3)
axes[1].plot(time_s/3600, scintillations, color="coral",     lw=0.5)
label = "Scintillations " + str(LOWCUT) + "-" + str(HIGHCUT) + " Hz"
axes[1].set(ylabel="Amplitude", title=label); axes[1].grid(alpha=0.3)
axes[-1].set_xlabel("Time (hours)")
plt.tight_layout(); plt.show()

## 4. CWT Spectrogram
Generalized Morse Wavelet with Synchrosqueezing for sharp frequency localisation.

In [ ]:
from scipy.interpolate import pchip_interpolate

# Upsample before CWT for better frequency resolution
new_fs = fs * PCHIP_FACTOR
t_up   = np.linspace(time_s[0], time_s[-1], len(time_s) * PCHIP_FACTOR)
sig_up = pchip_interpolate(time_s, scintillations, t_up)

spec = compute_cwt_spectrogram(
    sig_up, fs=new_fs, lowcut=LOWCUT, highcut=HIGHCUT,
    nv=CWT_NV, use_ssq=True  # use_ssq=False for very long signals (>14 h)
)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(spec.T, aspect="auto", origin="lower",
               extent=[time_s[0]/3600, time_s[-1]/3600, LOWCUT, HIGHCUT],
               cmap="viridis")
plt.colorbar(im, ax=ax, label="Power (dB)")
tgt = session["target"]
ax.set(xlabel="Time (hours)", ylabel="Frequency (Hz)",
       title=f"CWT Spectrogram -- {tgt} -- {CHANNEL}")
plt.tight_layout(); plt.show()
print("Spectrogram shape:", spec.shape, "(time x frequency)")

## Notes
- Run the full interactive GUI with: `uv run app.py`  
- Hyperparameters live in `core/config.py` and are editable live in the Settings dialog.  
- `use_ssq=True` gives sharp spectrograms on short signals via Synchrosqueezing.  
- Use `use_ssq=False` for very long signals (> 14 h) to avoid memory issues.